CREACIÓN DE MÁSCARAS

In [1]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from datetime import datetime
import pydicom
from pathlib import Path
import matplotlib.pyplot as plt
import os
import napari
import numpy as np
import cv2
import SimpleITK as sitk
import json

In [2]:
# Cargar carpeta DICOM con SimpleITK

def load_dicom_folder(path):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(path)
    reader.SetFileNames(dicom_names)
    image = reader.Execute()
    return image

# Convertir de SimpleITK a NumPy

def sitk_to_numpy(image_sitk):
    arr = sitk.GetArrayFromImage(image_sitk)  # (slices, h, w)
    return arr

# Crear dos máscaras de ejemplo

def CrearMascaras(imagen, circulos):
    """
    Crea una máscara con varios círculos.
    
    Parámetros:
        imagen: array 2D de la imagen
        circulos: lista de tuplas [(x, y, diametro), ...]
            x, y: posición del centro del círculo
            diametro: diámetro del círculo

    """
    h, w = imagen.shape
    mascara = np.zeros((h, w), dtype='uint8')

    for (x, y, diam) in circulos:
        radio = diam // 2
        cv2.circle(mascara, (x, y), radio, 255, -1)

    cv2.imshow("mascara", mascara)
    cv2.moveWindow("mascara", w, h)

    return mascara


if __name__ == "__main__":

    ct_path = r"C:\Users\gervi\OneDrive - Universidad Complutense de Madrid (UCM)\MASTER\SEGUNDO CUATRI\TFM\Imagenes prueba\CT"

    # Cargar volumen CT
    ct = load_dicom_folder(ct_path)

    # Convertir a NumPy
    np_img = sitk_to_numpy(ct)

    # Elegir slice central
    mid = np_img.shape[0] // 2
    img = np_img[mid]

    # Normalizar a uint8 para OpenCV
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')

    circulos = [
    (100, 100, 10),  
    (200, 100, 13),
    (300, 100, 17),
    (100, 200, 22),
    (200, 200, 28),
    (300, 200, 37)
]

    masc = CrearMascaras(img, circulos)

    cv2.waitKey(0)



In [2]:
import SimpleITK as sitk
import numpy as np
from scipy import ndimage
import cv2

# ---------- Funciones ----------
def load_dicom_folder(path):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(path)
    reader.SetFileNames(dicom_names)
    image = reader.Execute()
    return image

def sitk_to_numpy(image_sitk):
    return sitk.GetArrayFromImage(image_sitk)  # (slices, h, w)

def detectar_centros_slice(slice_img, umbral=200, min_size=20):
    """
    Detecta los centros de las esferas en un slice 2D usando umbral y etiquetado.
    
    Retorna lista de (y, x)
    """
    mask = slice_img > umbral
    labeled, num_features = ndimage.label(mask)

    centros = []
    for i in range(1, num_features + 1):
        coords = np.argwhere(labeled == i)
        if coords.shape[0] < min_size:
            continue
        y, x = coords.mean(axis=0)
        centros.append((int(y), int(x)))

    return centros

def crear_mascara_slice(slice_img, centros, diametros):
    """
    Crea máscara 2D usando centros detectados y diámetros conocidos.
    """
    h, w = slice_img.shape
    mascara = np.zeros((h, w), dtype=np.uint8)

    for (y, x), diam in zip(centros, diametros):
        radio = diam // 2
        cv2.circle(mascara, (x, y), radio, 255, -1)

    return mascara

# ---------- Programa principal ----------
if __name__ == "__main__":
    ct_path = r"C:\Users\gervi\OneDrive - Universidad Complutense de Madrid (UCM)\MASTER\SEGUNDO CUATRI\TFM\Imagenes prueba\CT"

    # Cargar volumen CT
    ct = load_dicom_folder(ct_path)
    np_img = sitk_to_numpy(ct)

    # Elegir slice central
    #mid = np_img.shape[0] // 2
    slice_img = np_img[114]

    # Diametros conocidos de las esferas
    diametros = [10, 13, 17, 22, 28, 37]

    # Detectar centros
    centros = detectar_centros_slice(slice_img, umbral=200, min_size=20)
    print("Centros detectados (y, x):", centros)

    if len(centros) != len(diametros):
        print("Advertencia: el número de centros detectados no coincide con los diámetros conocidos.")

    # Crear máscara 2D con radios exactos
    mascara = crear_mascara_slice(slice_img, centros, diametros)

    # Mostrar resultados
    cv2.imshow("Slice original", cv2.normalize(slice_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8))
    cv2.imshow("Mascara de esferas", mascara)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


Centros detectados (y, x): [(411, 273)]
Advertencia: el número de centros detectados no coincide con los diámetros conocidos.
